# Exploratory Data Analysis: NPPAD Dataset

Exploring the Nuclear Power Plant Accident Data (NPPAD) dataset:
- 97 operational parameters from a PWR simulator (PCTRAN)
- 18 operating conditions (1 normal + 17 accident types)

**Citation:** Qi, B., Xiao, X., Liang, J. et al. *Sci Data* 9, 766 (2022).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

RAW_DIR = Path('../data/raw/NuclearPowerPlantAccidentData')

## 1. Discover Available Data

In [ ]:
# Find all CSV files
csv_files = sorted(RAW_DIR.rglob('*.csv'))
print(f'Found {len(csv_files)} CSV files')

# Show directory structure
dirs = set(f.parent.relative_to(RAW_DIR) for f in csv_files)
for d in sorted(dirs):
    n = len(list(RAW_DIR.joinpath(d).glob('*.csv')))
    print(f'  {d}: {n} files')

## 2. Load Sample Data

In [ ]:
# Load one normal and one accident scenario to understand the schema
sample_file = csv_files[0]
df_sample = pd.read_csv(sample_file)
print(f'Sample file: {sample_file.relative_to(RAW_DIR)}')
print(f'Shape: {df_sample.shape}')
print(f'\nColumns ({len(df_sample.columns)}):')
print(df_sample.columns.tolist())
df_sample.head()

In [ ]:
# Summary statistics
df_sample.describe()

## 3. Parameter Distributions Under Normal Conditions

In [ ]:
# Load all normal operation data
normal_files = [f for f in csv_files if 'normal' in str(f).lower()]
if not normal_files:
    # Try first file as baseline
    normal_files = csv_files[:1]
    
df_normal = pd.concat([pd.read_csv(f) for f in normal_files], ignore_index=True)
numeric_cols = df_normal.select_dtypes(include=[np.number]).columns
print(f'Normal data: {len(df_normal)} timesteps, {len(numeric_cols)} numeric columns')

# Plot distributions of key parameters
key_params = numeric_cols[:12]  # First 12 parameters
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flat, key_params):
    df_normal[col].hist(ax=ax, bins=50, alpha=0.7)
    ax.set_title(col, fontsize=9)
plt.suptitle('Parameter Distributions - Normal Operation', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Normal vs Accident Comparison

In [ ]:
# Load a few accident scenarios for comparison
accident_files = [f for f in csv_files if 'normal' not in str(f).lower()][:5]

fig, axes = plt.subplots(len(accident_files), 1, figsize=(14, 4 * len(accident_files)))
if len(accident_files) == 1:
    axes = [axes]

param_col = numeric_cols[1] if len(numeric_cols) > 1 else numeric_cols[0]
time_col = numeric_cols[0]  # Usually TIME

for ax, f in zip(axes, accident_files):
    df_acc = pd.read_csv(f)
    ax.plot(df_normal[param_col].values[:len(df_acc)], label='Normal', alpha=0.7)
    ax.plot(df_acc[param_col].values, label=f.parent.name, alpha=0.7)
    ax.set_title(f'{f.parent.name} - {param_col}')
    ax.legend()

plt.suptitle('Normal vs Accident Scenarios', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
# Correlation matrix of parameters under normal operation
corr = df_normal[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr, cmap='RdBu_r', center=0, ax=ax,
            xticklabels=False, yticklabels=False)
ax.set_title('Parameter Correlation Matrix - Normal Operation')
plt.tight_layout()
plt.show()

# Find highly correlated pairs
high_corr = np.where(np.abs(corr.values) > 0.9)
pairs = [(corr.index[i], corr.columns[j], corr.values[i, j])
         for i, j in zip(*high_corr) if i < j]
print(f'\n{len(pairs)} highly correlated parameter pairs (|r| > 0.9):')
for p1, p2, r in sorted(pairs, key=lambda x: -abs(x[2]))[:10]:
    print(f'  {p1} <-> {p2}: r={r:.3f}')

## 6. Accident Divergence Timeline

How quickly do accident signatures diverge from normal operation?

In [ ]:
# For each accident file, compute MSE from normal baseline over time
normal_vals = df_normal[numeric_cols].values
normal_mean = normal_vals.mean(axis=0)
normal_std = normal_vals.std(axis=0) + 1e-8

fig, ax = plt.subplots(figsize=(14, 6))

for f in accident_files:
    df_acc = pd.read_csv(f)
    acc_vals = df_acc[numeric_cols].values
    # Z-score deviation from normal
    deviation = np.abs((acc_vals - normal_mean) / normal_std).mean(axis=1)
    ax.plot(deviation, label=f.parent.name, alpha=0.7)

ax.set_xlabel('Timestep')
ax.set_ylabel('Mean |Z-score| deviation from normal')
ax.set_title('Accident Divergence from Normal Operating Conditions')
ax.legend()
plt.tight_layout()
plt.show()